# Notebook 05 — Stockout & Overstock Risk Scoring

## Objective

This notebook converts demand forecasts into inventory decisions.

Using the weekly demand forecasts generated in Notebook 04, each SKU is classified according to its inventory risk.

The notebook produces:

- Stockout Risk
- Overstock Risk
- Recommended Action
- Estimated Rupee Value at Risk

The scoring logic is entirely rule-based and transparent, ensuring every recommendation can be easily explained to business stakeholders.

This notebook satisfies Deliverable D4 of Project FORESIGHT.

In [3]:
# ============================================================
# Imports
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path

In [4]:
# ============================================================
# Project Paths
# ============================================================

ROOT_DIR = Path.cwd().parent

DATA_DIR = ROOT_DIR / "data"

PROCESSED_DATA_DIR = DATA_DIR / "processed"

MODEL_DIR = ROOT_DIR / "models"

OUTPUT_DIR = ROOT_DIR / "outputs"

OUTPUT_DIR.mkdir(exist_ok=True)

print("Project Paths Ready")

Project Paths Ready


## Load Forecast Output

The forecast file produced in Notebook 04 contains weekly demand predictions for every SKU.

This dataset forms the basis for risk scoring.

In [5]:
# ============================================================
# Check Outputs Folder
# ============================================================

from pathlib import Path

for file in OUTPUT_DIR.glob("*"):
    print(file.name)

In [6]:
for file in MODEL_DIR.glob("*"):
    print(file.name)

In [7]:
from pathlib import Path

MODEL_DIR = ROOT_DIR / "models"

for f in MODEL_DIR.glob("*"):
    print(f.name)

In [8]:
# ============================================================
# Load Weekly Forecasts
# ============================================================

forecast_df = pd.read_parquet(
    PROCESSED_DATA_DIR / "weekly_forecasts.parquet"
)

print("Forecast Shape:", forecast_df.shape)

forecast_df.head()

Forecast Shape: (2030634, 7)


,item_id,store_id,year,week_of_year,units_sold,baseline_forecast,rf_forecast
0,HOUSEHOLD_1_447,CA_3,2011,4,0,0.0,0.0
1,HOUSEHOLD_1_447,CA_3,2011,5,0,0.0,0.0
2,HOUSEHOLD_1_447,CA_3,2011,6,0,0.0,0.0
3,HOUSEHOLD_1_447,CA_3,2011,7,0,0.0,0.0
4,HOUSEHOLD_1_447,CA_3,2011,8,0,0.0,0.0


In [9]:
forecast_df.columns.tolist()

['item_id',
 'store_id',
 'year',
 'week_of_year',
 'units_sold',
 'baseline_forecast',
 'rf_forecast']

# D4 Risk Scoring

The objective of this notebook is to convert demand forecasts into actionable inventory decisions.

For every SKU we will:

- estimate inventory position
- estimate future demand
- calculate stockout and overstock risk
- attach business actions
- quantify rupee impact

Unlike the forecasting model, this logic is fully transparent and explainable so planners can understand why every recommendation is produced.

In [10]:
# ============================================================
# Estimate Inventory Position
# ============================================================

forecast_df["estimated_inventory"] = (
    forecast_df["units_sold"] * 1.5
).round()

forecast_df.head()

,item_id,store_id,year,week_of_year,units_sold,baseline_forecast,rf_forecast,estimated_inventory
0,HOUSEHOLD_1_447,CA_3,2011,4,0,0.0,0.0,0.0
1,HOUSEHOLD_1_447,CA_3,2011,5,0,0.0,0.0,0.0
2,HOUSEHOLD_1_447,CA_3,2011,6,0,0.0,0.0,0.0
3,HOUSEHOLD_1_447,CA_3,2011,7,0,0.0,0.0,0.0
4,HOUSEHOLD_1_447,CA_3,2011,8,0,0.0,0.0,0.0


In [11]:
# ============================================================
# Safety Stock
# ============================================================

forecast_df["safety_stock"] = (
    forecast_df["rf_forecast"] * 0.30
).round()

In [12]:
# ============================================================
# Inventory Gap
# ============================================================

forecast_df["inventory_gap"] = (
    forecast_df["estimated_inventory"]
    - forecast_df["rf_forecast"]
)

In [13]:
# ============================================================
# Risk Classification
# ============================================================

conditions = [
    forecast_df["inventory_gap"] < 0,
    forecast_df["inventory_gap"] > forecast_df["rf_forecast"] * 0.50
]

choices = [
    "Stockout Risk",
    "Overstock Risk"
]

forecast_df["risk"] = np.select(
    conditions,
    choices,
    default="Healthy"
)

forecast_df["risk"].value_counts()

risk
Stockout Risk     853318
Overstock Risk    593552
Healthy           583764
Name: count, dtype: int64

In [14]:
# ============================================================
# Recommended Inventory Actions
# ============================================================

action_map = {
    "Stockout Risk": "Increase Reorder Quantity",
    "Overstock Risk": "Reduce Purchase / Apply Discount",
    "Healthy": "Maintain Current Inventory"
}

forecast_df["recommended_action"] = (
    forecast_df["risk"]
    .map(action_map)
)

forecast_df[
    ["risk", "recommended_action"]
].head()

,risk,recommended_action
0,Healthy,Maintain Current Inventory
1,Healthy,Maintain Current Inventory
2,Healthy,Maintain Current Inventory
3,Healthy,Maintain Current Inventory
4,Healthy,Maintain Current Inventory


In [15]:
# ============================================================
# Load Weekly Modeling Data for Price Information
# ============================================================

weekly_price_df = pd.read_parquet(
    PROCESSED_DATA_DIR / "weekly_chunks" / "weekly_chunk_001.parquet"
)

weekly_price_df.head()

,item_id,store_id,year,week_of_year,units_sold,sell_price,has_event,has_snap,is_weekend
0,FOODS_1_001,CA_1,2011,4,3,2.0,0,0,1
1,FOODS_1_001,CA_1,2011,5,9,2.0,1,1,1
2,FOODS_1_001,CA_1,2011,6,7,2.0,0,1,1
3,FOODS_1_001,CA_1,2011,7,10,2.0,1,1,1
4,FOODS_1_001,CA_1,2011,8,14,2.0,1,0,1


In [16]:
weekly_price_df.columns.tolist()

['item_id',
 'store_id',
 'year',
 'week_of_year',
 'units_sold',
 'sell_price',
 'has_event',
 'has_snap',
 'is_weekend']

In [17]:
# ============================================================
# Average Selling Price per SKU
# ============================================================

sku_price = (
    weekly_price_df
    .groupby("item_id", as_index=False)["sell_price"]
    .mean()
)

sku_price.head()

,item_id,sell_price
0,FOODS_1_001,2.000000
1,FOODS_1_002,7.920816
2,FOODS_1_003,2.880000
3,FOODS_1_004,NaN
4,FOODS_1_005,2.971915


In [18]:
# ============================================================
# Merge Price Information
# ============================================================

forecast_df = forecast_df.merge(
    sku_price,
    on="item_id",
    how="left"
)

forecast_df[["item_id", "sell_price"]].head()

,item_id,sell_price
0,HOUSEHOLD_1_447,NaN
1,HOUSEHOLD_1_447,NaN
2,HOUSEHOLD_1_447,NaN
3,HOUSEHOLD_1_447,NaN
4,HOUSEHOLD_1_447,NaN


In [19]:
# ============================================================
# Rupee Value at Stake
# ============================================================

# Sales at risk from stockout
forecast_df["sales_at_risk_rs"] = np.where(
    forecast_df["risk"] == "Stockout Risk",
    abs(forecast_df["inventory_gap"]) * forecast_df["sell_price"],
    0
)

# Locked capital from overstock
forecast_df["locked_capital_rs"] = np.where(
    forecast_df["risk"] == "Overstock Risk",
    forecast_df["inventory_gap"] * forecast_df["sell_price"],
    0
)

forecast_df[
    [
        "risk",
        "sales_at_risk_rs",
        "locked_capital_rs"
    ]
].head()

,risk,sales_at_risk_rs,locked_capital_rs
0,Healthy,0.0,0.0
1,Healthy,0.0,0.0
2,Healthy,0.0,0.0
3,Healthy,0.0,0.0
4,Healthy,0.0,0.0


In [20]:
# ============================================================
# Business Impact Summary
# ============================================================

total_sales_risk = forecast_df["sales_at_risk_rs"].sum()

total_locked_capital = forecast_df["locked_capital_rs"].sum()

print(f"Estimated Sales at Risk : ₹{total_sales_risk:,.0f}")

print(f"Estimated Locked Capital: ₹{total_locked_capital:,.0f}")

Estimated Sales at Risk : ₹4,505,995
Estimated Locked Capital: ₹9,475,885


In [21]:
# ============================================================
# Decision Grid
# ============================================================

decision_map = {
    "Stockout Risk": {
        "Priority": "High",
        "Decision": "Reorder Immediately"
    },
    "Overstock Risk": {
        "Priority": "Medium",
        "Decision": "Reduce Purchasing / Launch Promotion"
    },
    "Healthy": {
        "Priority": "Low",
        "Decision": "Maintain Current Inventory"
    }
}

forecast_df["priority"] = forecast_df["risk"].map(
    lambda x: decision_map[x]["Priority"]
)

forecast_df["decision"] = forecast_df["risk"].map(
    lambda x: decision_map[x]["Decision"]
)

forecast_df[
    [
        "risk",
        "priority",
        "decision"
    ]
].head()

,risk,priority,decision
0,Healthy,Low,Maintain Current Inventory
1,Healthy,Low,Maintain Current Inventory
2,Healthy,Low,Maintain Current Inventory
3,Healthy,Low,Maintain Current Inventory
4,Healthy,Low,Maintain Current Inventory


In [22]:
# ============================================================
# Decision Summary
# ============================================================

decision_summary = (

    forecast_df

    .groupby(
        [
            "risk",
            "priority",
            "decision"
        ]
    )

    .agg(
        sku_count=("item_id","count"),
        sales_at_risk=("sales_at_risk_rs","sum"),
        locked_capital=("locked_capital_rs","sum")
    )

    .reset_index()

)

decision_summary

,risk,priority,decision,sku_count,sales_at_risk,locked_capital
0,Healthy,Low,Maintain Current Inventory,583764,0.000000e+00,0.000000e+00
1,Overstock Risk,Medium,Reduce Purchasing / Launch Promotion,593552,0.000000e+00,9.475885e+06
2,Stockout Risk,High,Reorder Immediately,853318,4.505995e+06,0.000000e+00


In [23]:
# ============================================================
# Save Final Risk Scoring Output
# ============================================================

OUTPUT_DIR = ROOT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

forecast_df.to_csv(
    OUTPUT_DIR / "risk_scoring_results.csv",
    index=False
)

decision_summary.to_csv(
    OUTPUT_DIR / "decision_summary.csv",
    index=False
)

print("Risk scoring files exported successfully.")

Risk scoring files exported successfully.


In [25]:
OUTPUT_DIR = ROOT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

forecast_df.to_csv(
    OUTPUT_DIR / "risk_scoring_results.csv",
    index=False
)

decision_summary.to_csv(
    OUTPUT_DIR / "decision_summary.csv",
    index=False
)

print("Risk scoring outputs saved successfully.")

Risk scoring outputs saved successfully.


In [24]:
# ============================================================
# Notebook 05 Validation
# ============================================================

print("="*55)
print("PROJECT FORESIGHT - NOTEBOOK 05 VALIDATION")
print("="*55)

print()

print("Forecast Records :", len(forecast_df))

print()

print("Risk Distribution")
print(forecast_df["risk"].value_counts())

print()

print(f"Estimated Sales at Risk : ₹{forecast_df['sales_at_risk_rs'].sum():,.0f}")

print(f"Estimated Locked Capital : ₹{forecast_df['locked_capital_rs'].sum():,.0f}")

print()

print("Decision Summary")

display(decision_summary)

print()

print("Notebook 05 Completed Successfully!")

PROJECT FORESIGHT - NOTEBOOK 05 VALIDATION

Forecast Records : 2030634

Risk Distribution
risk
Stockout Risk     853318
Overstock Risk    593552
Healthy           583764
Name: count, dtype: int64

Estimated Sales at Risk : ₹4,505,995
Estimated Locked Capital : ₹9,475,885

Decision Summary


,risk,priority,decision,sku_count,sales_at_risk,locked_capital
0,Healthy,Low,Maintain Current Inventory,583764,0.000000e+00,0.000000e+00
1,Overstock Risk,Medium,Reduce Purchasing / Launch Promotion,593552,0.000000e+00,9.475885e+06
2,Stockout Risk,High,Reorder Immediately,853318,4.505995e+06,0.000000e+00



Notebook 05 Completed Successfully!


# Notebook Summary

This notebook converts demand forecasts into actionable inventory recommendations.

Outputs produced:

- SKU-level stockout risk
- SKU-level overstock risk
- Recommended inventory action
- Estimated sales at risk (₹)
- Estimated locked capital (₹)
- Decision priority (High / Medium / Low)

These outputs satisfy Deliverable D4 of Project FORESIGHT and provide the operational decision layer built on top of the demand forecasting model.